In [ ]:
from typing import List
import numpy as np
import pandas as pd
import io
import tellurium as te
import re 
from sklearn.preprocessing import StandardScaler

In [ ]:
def simulate(model_str: str) -> dict:
    r = te.loada(model_str)
    sim_result = r.simulate()
    sim_output = sim_result[:, 1:]  # drop time column
    return {"data":sim_output, "species":r.getFloatingSpeciesNames()}

def encode(model_str: str) -> np.ndarray:
    data = np.asarray(simulate(model_str).get("data"))
    scaler = StandardScaler()
    clip_range = (-5, 5)
    num_bins = 4096
    bin_edges = np.linspace(clip_range[0], clip_range[1], num_bins + 1)

    normed = scaler.fit_transform(data.reshape(-1, 1)).flatten()
    normed = np.clip(normed, *clip_range)
    token_ids = np.digitize(normed, bin_edges) - 1
    return token_ids


def format_prompt(sample: dict) -> str:
    data = sample["series"]
    prev_data = sample["prev_data"]
    fin_str = ""
    for series in data:
        encoded = encode(np.array(series))
        fin_str += " ".join(map(str, encoded)) + "\n"

    prompt = f"""
    You are a model tasked with recovering the biological network corresponding 
    to the time series data provided.
    Previous info: {prev_data}
    RETURN ONLY THE MODEL IN ANTIMONY FORMAT. YOU MUST ESTIMATE ALL PARAMETERS. 
    YOU MUST COME UP WITH NEW REACTIONS

    Time Series Data:
    {fin_str}
    """
    return prompt.strip()

def get_prev_data_and_format(data: dict, model_str: str) -> str:
    prev_species = data.get("species")
    pattern = r'r^[A-Za-z0-9]+:\s*.+?;,+$'
    matches = re.findall(pattern, model_str, flags=re.MULTILINE)
    reactions = matches[:2] #get first two reactions 
    return f"""
    {reactions} \n {prev_species}
    """

